# 02 - Pré-processamento

Camada **Silver**: limpeza e padronização dos dados da camada Bronze (`dados/bronze/`), gerando os dados tratados em `dados/silver/`.

In [1]:
import sys
from pathlib import Path

# Raiz do projeto (notebooks/ -> raiz do repositorio). Fixado aqui porque o
# kernel do Jupyter roda com cwd = pasta do notebook, nao a raiz do projeto.
RAIZ_PROJETO = Path.cwd().parent
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.append(str(RAIZ_PROJETO))

from src.preprocessamento.preprocessamento import Preprocessamento

In [2]:
import logging

# Idem ao notebook de ingestão: os logs INFO detalhados (nulos por coluna,
# linhas descartadas por período) continuam disponíveis via
# `python -m src.preprocessamento.preprocessamento` — aqui deixamos só
# avisos/erros e usamos as células de conferência abaixo.
logging.getLogger("src.preprocessamento.preprocessamento").setLevel(logging.WARNING)

etapa_preprocessamento = Preprocessamento(
    caminho_entrada=RAIZ_PROJETO / "dados" / "bronze",
    caminho_saida=RAIZ_PROJETO / "dados" / "silver",
)
resultados_preprocessamento = etapa_preprocessamento.executar()
print(f"Pré-processamento concluído: {len(resultados_preprocessamento)} período(s) processado(s).")

Pré-processamento concluído: 12 período(s) processado(s).


## Resumo do pré-processamento

Reconciliação Bronze → Silver por período (RNF-05): quantas linhas entraram,
quantas saíram e quantas foram descartadas (identificação nula/duplicada —
não inclui filtro de regra de negócio, que fica para a Transformação/Gold).

In [3]:
import pandas as pd

resumo_preprocessamento = (
    pd.DataFrame([r.__dict__ for r in resultados_preprocessamento])
    .assign(
        periodo=lambda df: df["ano"].astype(str) + "Q" + df["trimestre"].astype(str),
        descartadas=lambda df: df["linhas_entrada"] - df["linhas_saida"],
    )
    .set_index("periodo")[["linhas_entrada", "linhas_saida", "descartadas", "duracao_segundos"]]
    .round({"duracao_segundos": 1})
)

total = resumo_preprocessamento[["linhas_entrada", "linhas_saida", "descartadas"]].sum()
print(f"Total consolidado: entrada={total['linhas_entrada']:,} saida={total['linhas_saida']:,} descartadas={total['descartadas']:,}".replace(",", "."))
resumo_preprocessamento.style.bar(subset=["descartadas"], color="#cf222e")

Total consolidado: entrada=5.748.375 saida=5.748.375 descartadas=0


C:\Users\Gui\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\io\formats\style.py:4438: RuntimeWarning: invalid value encountered in scalar divide
  end = (x - left) / (right - left)


,linhas_entrada,linhas_saida,descartadas,duracao_segundos
periodo,,,,
2023Q1,473335,473335,0,10.300000
2023Q2,474575,474575,0,9.900000
2023Q3,479873,479873,0,10.000000
2023Q4,473206,473206,0,10.000000
2024Q1,481349,481349,0,10.100000
2024Q2,479986,479986,0,10.100000
2024Q3,479778,479778,0,10.000000
2024Q4,469334,469334,0,9.800000
2025Q1,472367,472367,0,9.800000


## Conferência da camada Silver

Carrega o parquet consolidado e inspeciona: dimensões, amostra de linhas,
tipos e o percentual de nulo por coluna (esperado ser alto em `VD4009` e
colegas — são variáveis de trabalho, não preenchidas para quem está fora da
força de trabalho; isso é tratado na Transformação/Gold, não aqui).

In [4]:
dados_silver = pd.read_parquet(RAIZ_PROJETO / "dados" / "silver" / "dados_silver.parquet")
print(f"Dimensoes: {dados_silver.shape[0]:,} linhas x {dados_silver.shape[1]} colunas".replace(",", "."))
dados_silver.head(10)

Dimensoes: 5.748.375 linhas x 29 colunas


,Ano,Trimestre,UF,UPA,V1008,V1014,V2003,VD4009,V4019,VD4012,...,V3002,VD4010,VD4011,V4018,V4025,V4040,VD4031,V1028,VD4016,VD4017
0,2023,1,11,110000016,1,10,1,9.0,2.0,2.0,...,2.0,2.0,9.0,1.0,NaN,4.0,40.0,186.952067,5000.0,5000.0
1,2023,1,11,110000016,1,10,2,NaN,NaN,NaN,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,186.952067,NaN,NaN
2,2023,1,11,110000016,2,10,1,7.0,NaN,1.0,...,2.0,9.0,3.0,NaN,2.0,4.0,40.0,96.058559,3500.0,3500.0
3,2023,1,11,110000016,3,10,1,NaN,NaN,NaN,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,172.249276,NaN,NaN
4,2023,1,11,110000016,3,10,2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,172.249276,NaN,NaN
5,2023,1,11,110000016,4,10,1,NaN,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,131.868954,NaN,NaN
6,2023,1,11,110000016,4,10,2,NaN,NaN,NaN,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,131.868954,NaN,NaN
7,2023,1,11,110000016,4,10,3,7.0,NaN,1.0,...,2.0,9.0,2.0,NaN,2.0,4.0,40.0,131.868954,3500.0,3500.0
8,2023,1,11,110000016,5,10,1,NaN,NaN,NaN,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,180.537822,NaN,NaN
9,2023,1,11,110000016,5,10,2,9.0,2.0,2.0,...,2.0,10.0,5.0,1.0,NaN,4.0,44.0,180.537822,3000.0,3000.0


In [5]:
percentual_nulo = (dados_silver.isna().mean() * 100).round(1).rename("% nulo").to_frame()
percentual_nulo.style.bar(subset=["% nulo"], color="#9a6700", vmin=0, vmax=100)

,% nulo
Ano,0.000000
Trimestre,0.000000
UF,0.000000
UPA,0.000000
V1008,0.000000
V1014,0.000000
V2003,0.000000
VD4009,56.100000
V4019,86.100000
VD4012,56.100000


In [6]:
# Confere se os 12 períodos (2023-2025, 4 trimestres) realmente foram
# consolidados no arquivo unico — cada linha deve ter linhas > 0.
contagem_por_periodo = (
    dados_silver.groupby(["Ano", "Trimestre"]).size().rename("linhas").to_frame()
)
contagem_por_periodo

linhas
Ano  Trimestre        
2023 1          473335
     2          474575
     3          479873
     4          473206
2024 1          481349
     2          479986
     3          479778
     4          469334
2025 1          472367
     2          475590
     3          490488
     4          498494